<!-- track-identity-card -->
# Corner detection and phase segmentation

| | |
|---|---|
| Pipeline step | `02_corner_segmentation.ipynb` |
| Manuscript section | 3.2 |
| Copied from | `notebooks/NB08_corner_segmentation_v4.ipynb` |
| Source sha256 | `fecd3494fe21598363146abb0f75582d` |

**Reads**

- `data/processed/<tier>/<track>*.parquet`

**Writes**

- `data/features/driver_corner_matrix_<track>.parquet`
- `data/features/driver_meta_<track>.parquet`

Defines segment_corner, detect_steer_column, wavg, build_driver_matrix_v3. Source of the corner inventory reported in Section 3.2.

> Copied verbatim from the working notebook. The identity card above is the only addition; no code cell was modified.


# 08 — Corner Phase Segmentation v4: Exit Speed Fix
**Sim Racing Telemetry Analysis — MSc Thesis**

## v3 -> v4 Degisiklikleri

| Konu | v3 | v4 |
|------|----|----|
| Exit speed fallback | `apex_speed` kopyasi | **3-katmanli bagimsiz tespit** |
| Yeni parametre | - | `EXIT_FALLBACK_M = 50` |
| Yeni kolon | - | `seg_exit_method` |
| Motivasyon | `apex_std <-> exit_std` r=1.000 | Bagimsiz olcum, r < 1.0 bekleniyor |

## Sorun (v3)
```
exit_speed = apex_speed  # fallback: cikis bulunamazsa kopyalaniyor
```
## Cozum (v4): 3-Katmanli Exit Tespiti
```
1. Birincil : throttle > 0.3 VE speed > apex  (mevcut)
2. Ikincil  : throttle > 0.3                   (hiz kosulu yok)
3. Yedek    : apex + 50m mesafedeki hiz         (bagimsiz)
```


In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import savgol_filter
import warnings
warnings.filterwarnings('ignore')

PROJECT      = TRACK_ROOT
PROCESSED    = PROJECT / "data" / "processed" / "acgym_sessions"
FEATURES_DIR = PROJECT / "data" / "features"

TRACKS = ['monza', 'barcelona', 'red_bull_ring']

# Segmentasyon parametreleri
SEARCH_BACK_M      = 300
SEARCH_FWD_M       = 200
TRAIL_BRAKE_ZONE   = 0.40
TRAIL_MIN_PRESSURE = 0.03
MIN_APEX_SPEED     = 20.0   # km/h
MIN_ENTRY_SPEED    = 50.0   # km/h
MC_RADIUS_M        = 15     # mid-corner faz yaricapi (apex +/-15m)
EXIT_FALLBACK_M    = 50     # v4: exit fallback mesafesi (apex + N metre)

print("Config OK")
print(f"  MIN_APEX_SPEED  = {MIN_APEX_SPEED} km/h")
print(f"  MIN_ENTRY_SPEED = {MIN_ENTRY_SPEED} km/h")
print(f"  MC_RADIUS       = {MC_RADIUS_M} m")
print(f"  EXIT_FALLBACK_M = {EXIT_FALLBACK_M} m")


## ADIM 1 — Veri Yükleme

In [ ]:
loaded = {}

for track_name in TRACKS:
    corners_path = FEATURES_DIR / f"{track_name}_corners_v3.parquet"
    if not corners_path.exists():
        print(f"❌ {track_name}: corners_v3 bulunamadı")
        continue
    
    corners_v3 = pd.read_parquet(corners_path)
    
    # Telemetri dosyalarını bul ve birleştir
    tele_files = sorted(PROCESSED.glob(f"{track_name}*.parquet"))
    if not tele_files:
        print(f"❌ {track_name}: telemetri bulunamadı")
        continue
    
    tele_all = pd.concat([pd.read_parquet(f) for f in tele_files], ignore_index=True)
    
    # NaN sürücü filtresi
    if 'driver_id' in tele_all.columns:
        n_drivers = tele_all['driver_id'].nunique()
    else:
        n_drivers = 1
    
    loaded[track_name] = {
        'corners_v3': corners_v3,
        'tele_all': tele_all,
    }
    print(f"✅ {track_name}: {len(corners_v3)} viraj, {n_drivers} sürücü, {len(tele_all):,} satır")

print(f"\nYüklenen: {list(loaded.keys())}")

## ADIM 2 — Steering Kolon Tespiti

In [ ]:
def detect_steer_column(df):
    """
    Direksiyon veya lateral G kolonu otomatik tespit.
    
    Returns: (mode, column_name)
        mode: 'steer' | 'g_lat' | 'speed_proxy'
    """
    cols = {c: c.lower() for c in df.columns}
    
    # 1. Steering angle
    for c, cl in cols.items():
        if any(k in cl for k in ['steerangle', 'steering_angle', 'steer_angle', 'wheel_angle']):
            return ('steer', c)
    for c, cl in cols.items():
        if 'steer' in cl and 'error' not in cl:
            return ('steer', c)
    
    # 2. Lateral G
    for c, cl in cols.items():
        if any(k in cl for k in ['g_lat', 'glat', 'lateral_g', 'accg_y', 'accel_lat']):
            return ('g_lat', c)
    
    # 3. Fallback
    return ('speed_proxy', None)

# Tespit
for track_name, data in loaded.items():
    mode, col = detect_steer_column(data['tele_all'])
    data['steer_info'] = (mode, col)
    print(f"  {track_name}: mode={mode}, column={col}")

## ADIM 3 — Segmentasyon Fonksiyonları

### Tasarım prensibi
- `segment_corner()` → v2'den aynen korunur
- `segment_corner_phases()` → YENİ, 5-faz detayı üretir
- `enrich_corners()` → her ikisini çağırır, birleştirir

In [ ]:
# === segment_corner v4: 3-katmanli exit tespiti ===

def segment_corner(df_lap, apex_lapdist, difficulty):
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values   if 'accStatus'   in df_lap.columns else np.zeros(len(df_lap))
    dist  = df_lap['LapDist'].values

    apex_idx = int(np.argmin(np.abs(dist - apex_lapdist)))
    apex_speed = float(speed[apex_idx])

    # Entry: brake baslangici geriye dogru ara
    back_limit = max(0, apex_idx - int(SEARCH_BACK_M / 2))
    entry_idx  = apex_idx
    for j in range(apex_idx - 1, back_limit, -1):
        if brake[j] < 0.05 and speed[j] > speed[apex_idx]:
            entry_idx = j
            break

    entry_speed = float(speed[entry_idx])
    if entry_speed < MIN_ENTRY_SPEED or apex_speed < MIN_APEX_SPEED:
        return _empty_seg(apex_lapdist, apex_speed)

    # ====== EXIT: 3-katmanli tespit (v4) ======
    fwd_limit = min(len(speed) - 1, apex_idx + int(SEARCH_FWD_M / 2))
    exit_idx    = apex_idx
    exit_method = 'none'

    # Katman 1: throttle > 0.3 VE speed > apex (en kesin)
    for j in range(apex_idx + 1, fwd_limit):
        if acc[j] > 0.3 and speed[j] > apex_speed:
            exit_idx = j
            exit_method = 'throttle_full'
            break

    # Katman 2: sadece throttle > 0.3 (hiz kosulu kaldirildi)
    if exit_idx == apex_idx:
        for j in range(apex_idx + 1, fwd_limit):
            if acc[j] > 0.3:
                exit_idx = j
                exit_method = 'throttle_only'
                break

    # Katman 3: sabit mesafe fallback (apex_speed den bagimsiz)
    if exit_idx == apex_idx:
        fb_target = dist[apex_idx] + EXIT_FALLBACK_M
        fb_idx = int(np.argmin(np.abs(dist - fb_target)))
        exit_idx = min(fb_idx, fwd_limit)
        exit_method = 'distance_fallback'

    exit_speed = float(speed[exit_idx])
    # ====== EXIT v4 sonu ======

    entry_dist_v = float(dist[entry_idx])
    exit_dist_v  = float(dist[exit_idx])
    braking_dist = float(dist[apex_idx] - dist[entry_idx]) if entry_idx < apex_idx else 0.0
    speed_loss_eff = (apex_speed / entry_speed) if entry_speed > 0 else 0.0

    # Coasting
    coast_start = apex_idx
    for j in range(apex_idx, fwd_limit):
        if brake[j] < 0.05:
            coast_start = j
            break
    coast_end = coast_start
    for j in range(coast_start, fwd_limit):
        if acc[j] > 0.1:
            coast_end = j
            break
    coasting_dist = float(dist[coast_end] - dist[coast_start]) if coast_end > coast_start else 0.0

    # Trail braking
    if braking_dist > 10:
        trail_zone_start = entry_idx + int((apex_idx - entry_idx) * (1 - TRAIL_BRAKE_ZONE))
        trail_pressures  = brake[trail_zone_start:apex_idx]
        trail_active     = trail_pressures.mean() > TRAIL_MIN_PRESSURE if len(trail_pressures) > 0 else False
        trail_pressure   = float(trail_pressures.mean()) if len(trail_pressures) > 0 else 0.0
    else:
        trail_active   = False
        trail_pressure = 0.0

    avg_brake = float(brake[entry_idx:apex_idx].mean()) if apex_idx > entry_idx else 0.0

    return {
        'seg_entry_dist': entry_dist_v, 'seg_apex_dist': float(dist[apex_idx]),
        'seg_exit_dist': exit_dist_v,
        'seg_entry_speed': entry_speed, 'seg_apex_speed': apex_speed,
        'seg_exit_speed': exit_speed,
        'seg_exit_method': exit_method,
        'seg_braking_dist': braking_dist, 'seg_trail_braking': trail_active,
        'seg_trail_pressure': trail_pressure, 'seg_avg_brake_pressure': avg_brake,
        'seg_coasting_dist': coasting_dist, 'seg_speed_loss_eff': speed_loss_eff,
        'seg_corner_width': exit_dist_v - entry_dist_v,
        'seg_entry_valid': True,
    }


def _empty_seg(apex_lapdist, apex_speed=np.nan):
    keys = [
        'seg_entry_dist', 'seg_apex_dist', 'seg_exit_dist',
        'seg_entry_speed', 'seg_apex_speed', 'seg_exit_speed',
        'seg_exit_method',
        'seg_braking_dist', 'seg_trail_braking', 'seg_trail_pressure',
        'seg_avg_brake_pressure', 'seg_coasting_dist',
        'seg_speed_loss_eff', 'seg_corner_width',
        'phase_slb_dist', 'phase_slb_decel_rate',
        'phase_ce_dist', 'phase_ce_brake_at_turnin',
        'phase_mc_speed_ratio', 'phase_mc_lateral_signal',
        'phase_cex_throttle_lag', 'phase_cex_accel_rate',
        'phase_turnin_method',
    ]
    result = {k: np.nan for k in keys}
    result['seg_apex_dist'] = float(apex_lapdist)
    result['seg_apex_speed'] = float(apex_speed) if not np.isnan(apex_speed) else np.nan
    result['seg_exit_method'] = 'skipped'
    result['seg_entry_valid'] = False
    result['seg_trail_braking'] = False
    return result


def votes_to_confidence(n):
    return {0: 0.0, 1: 0.3, 2: 0.6, 3: 0.85, 4: 1.0}.get(int(n), 0.5)


print("segment_corner() v4 hazir — 3-katmanli exit tespiti aktif")


## ADIM 3b — Vidigal 5-Faz State Machine (YENİ)

In [ ]:
def segment_corner_phases(df_lap, seg, steer_info):
    """
    Vidigal (2023) ilhamıyla 5-faz state machine.
    Mevcut seg_* sonuçlarını genişletir — onlara dokunmaz.
    
    Döndürür: dict — phase_* prefixli 9 kolon
    """
    mode, steer_col = steer_info
    
    if not seg.get('seg_entry_valid', False) or pd.isna(seg.get('seg_entry_dist')):
        return {
            'phase_slb_dist': np.nan, 'phase_slb_decel_rate': np.nan,
            'phase_ce_dist': np.nan, 'phase_ce_brake_at_turnin': np.nan,
            'phase_mc_speed_ratio': np.nan, 'phase_mc_lateral_signal': np.nan,
            'phase_cex_throttle_lag': np.nan, 'phase_cex_accel_rate': np.nan,
            'phase_turnin_method': 'skipped',
        }
    
    dist  = df_lap['LapDist'].values
    speed = df_lap['speed_kmh'].values
    brake = df_lap['brakeStatus'].values if 'brakeStatus' in df_lap.columns else np.zeros(len(df_lap))
    acc   = df_lap['accStatus'].values if 'accStatus' in df_lap.columns else np.zeros(len(df_lap))
    
    entry_dist = seg['seg_entry_dist']
    apex_dist  = seg['seg_apex_dist']
    exit_dist  = seg['seg_exit_dist']
    
    entry_idx = int(np.argmin(np.abs(dist - entry_dist)))
    apex_idx  = int(np.argmin(np.abs(dist - apex_dist)))
    exit_idx  = int(np.argmin(np.abs(dist - exit_dist)))
    
    # ── Turn-in noktası tespiti ──
    turnin_idx = entry_idx + int((apex_idx - entry_idx) * 0.4)  # default fallback
    turnin_method = 'speed_proxy_40pct'
    
    if mode == 'steer' and steer_col in df_lap.columns:
        steer = np.abs(df_lap[steer_col].values)
        region = steer[entry_idx:apex_idx]
        if len(region) > 5:
            wlen = min(11, len(region))
            if wlen % 2 == 0:
                wlen -= 1
            if wlen >= 3:
                smoothed = savgol_filter(region, wlen, min(2, wlen - 1))
            else:
                smoothed = region
            # Turn-in: direksiyon açısının threshold'u aştığı ilk nokta
            threshold = smoothed.max() * 0.15
            for k, val in enumerate(smoothed):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'steer_threshold'
                    break
    
    elif mode == 'g_lat' and steer_col in df_lap.columns:
        glat = np.abs(df_lap[steer_col].values)
        region = glat[entry_idx:apex_idx]
        if len(region) > 5:
            threshold = region.max() * 0.20
            for k, val in enumerate(region):
                if val > threshold:
                    turnin_idx = entry_idx + k
                    turnin_method = 'g_lat_threshold'
                    break
    
    # Sınır kontrolü
    turnin_idx = max(entry_idx + 1, min(turnin_idx, apex_idx - 1))
    
    # ── MC fazı: apex ±MC_RADIUS_M ──
    mc_start = int(np.argmin(np.abs(dist - (apex_dist - MC_RADIUS_M))))
    mc_end   = int(np.argmin(np.abs(dist - (apex_dist + MC_RADIUS_M))))
    mc_start = max(mc_start, turnin_idx)
    mc_end   = min(mc_end, exit_idx)
    
    # ── Faz metrikleri hesapla ──
    
    # SLB: entry → turnin
    slb_dist = float(dist[turnin_idx] - dist[entry_idx]) if turnin_idx > entry_idx else 0.0
    slb_speed_drop = float(speed[entry_idx] - speed[turnin_idx])
    slb_decel = slb_speed_drop / max(slb_dist, 1.0) * (1000.0 / 3600.0)  # m/s² yaklaşık
    
    # CE: turnin → mc_start
    ce_dist = float(dist[mc_start] - dist[turnin_idx]) if mc_start > turnin_idx else 0.0
    ce_brake_at_turnin = float(brake[turnin_idx]) if turnin_idx < len(brake) else 0.0
    
    # MC: apex çevresi
    mc_speeds = speed[mc_start:mc_end+1]
    mc_speed_ratio = float(mc_speeds.min() / mc_speeds.max()) if len(mc_speeds) > 1 and mc_speeds.max() > 0 else 1.0
    
    mc_lateral = 0.0
    if mode == 'g_lat' and steer_col in df_lap.columns:
        mc_lateral = float(np.abs(df_lap[steer_col].values[mc_start:mc_end+1]).mean())
    elif mode == 'steer' and steer_col in df_lap.columns:
        mc_lateral = float(np.abs(df_lap[steer_col].values[mc_start:mc_end+1]).mean())
    
    # CEX: apex → exit (throttle lag)
    throttle_lag = 0.0
    for j in range(apex_idx, exit_idx):
        if acc[j] > 0.3:
            throttle_lag = float(dist[j] - dist[apex_idx])
            break
    
    cex_region = speed[apex_idx:exit_idx+1]
    if len(cex_region) > 1:
        cex_accel = float(cex_region[-1] - cex_region[0]) / max(float(dist[exit_idx] - dist[apex_idx]), 1.0)
    else:
        cex_accel = 0.0
    
    return {
        'phase_slb_dist': slb_dist,
        'phase_slb_decel_rate': slb_decel,
        'phase_ce_dist': ce_dist,
        'phase_ce_brake_at_turnin': ce_brake_at_turnin,
        'phase_mc_speed_ratio': mc_speed_ratio,
        'phase_mc_lateral_signal': mc_lateral,
        'phase_cex_throttle_lag': throttle_lag,
        'phase_cex_accel_rate': cex_accel,
        'phase_turnin_method': turnin_method,
    }


print("segment_corner_phases() hazır — Vidigal 5-faz ✅")

## ADIM 3c — Enrich Corners (Birleşik)

In [ ]:
def enrich_corners(corners_v3_df, df_single_driver, steer_info, track_name=None):
    segs = []
    for _, row in corners_v3_df.iterrows():
        seg = segment_corner(
            df_lap       = df_single_driver,
            apex_lapdist = float(row['apex_dist']),
            difficulty   = float(row.get('difficulty_score', 0.5)),
        )
        # Faz detayları ekle
        phase = segment_corner_phases(df_single_driver, seg, steer_info)
        seg.update(phase)
        segs.append(seg)
    
    enriched = corners_v3_df.reset_index(drop=True).join(pd.DataFrame(segs))
    enriched['confidence'] = enriched['n_votes'].apply(votes_to_confidence)
    enriched.loc[enriched['seg_entry_dist'].isna(), 'confidence'] = 0.0
    
    # MIN_APEX_SPEED filtresi
    bad_apex = enriched['seg_apex_speed'] < MIN_APEX_SPEED
    if bad_apex.any():
        enriched.loc[bad_apex, 'confidence'] = 0.0
    
    # entry > apex sanity check
    bad_entry = enriched['seg_entry_speed'] < enriched['seg_apex_speed']
    if bad_entry.any():
        enriched.loc[bad_entry, 'confidence'] = 0.0
    
    if track_name:
        ok  = (enriched['confidence'] > 0).sum()
        print(f"  [{track_name}] {ok}/{len(enriched)} viraj geçerli")
    
    return enriched


print("enrich_corners() hazır — segment + phases birleşik ✅")

## ADIM 4 — Tüm Pistleri İşle

In [ ]:
all_enriched = {}

for track_name, data in loaded.items():
    print(f"\n{'='*50}")
    print(f"  {track_name.upper()}")
    print(f"{'='*50}")
    
    corners_v3  = data['corners_v3']
    tele_all    = data['tele_all']
    steer_info  = data['steer_info']
    driver_ids  = sorted(tele_all['driver_id'].unique()) if 'driver_id' in tele_all.columns else ['all']
    
    track_results = []
    for driver_id in driver_ids:
        df_drv = tele_all[tele_all['driver_id'] == driver_id].reset_index(drop=True) \
                 if 'driver_id' in tele_all.columns else tele_all.copy()
        
        required = ['speed_kmh', 'LapDist', 'brakeStatus', 'accStatus']
        missing  = [c for c in required if c not in df_drv.columns]
        if missing:
            continue
        
        enriched = enrich_corners(corners_v3, df_drv, steer_info)
        enriched['driver_id'] = driver_id
        track_results.append(enriched)
    
    if track_results:
        all_enriched[track_name] = pd.concat(track_results, ignore_index=True)
        
        # Turnin method dağılımı
        methods = all_enriched[track_name]['phase_turnin_method'].value_counts()
        print(f"  Turn-in method dağılımı:")
        for m, c in methods.items():
            print(f"    {m}: {c}")
        
        # Phase kolon özeti
        phase_cols = [c for c in all_enriched[track_name].columns if c.startswith('phase_')]
        print(f"  Phase kolonları: {len(phase_cols)} adet")

## ADIM 5 — Driver Matrix (Faz Özellikleri Dahil)

In [ ]:
def wavg(values, weights):
    v    = pd.to_numeric(pd.Series(values), errors='coerce').values
    w    = np.array(weights, dtype=float)
    mask = ~np.isnan(v) & (w > 0)
    return float(np.average(v[mask], weights=w[mask])) if mask.sum() > 0 else np.nan


def build_driver_matrix_v3(track_name):
    if track_name not in loaded or track_name not in all_enriched:
        return None, None
    
    corners_v3 = loaded[track_name]['corners_v3']
    tele_all   = loaded[track_name]['tele_all']
    steer_info = loaded[track_name]['steer_info']
    driver_ids = sorted(tele_all['driver_id'].unique()) if 'driver_id' in tele_all.columns else ['all']
    
    all_records  = []
    meta_records = []
    
    for driver_id in driver_ids:
        df_drv = tele_all[tele_all['driver_id'] == driver_id].reset_index(drop=True) \
                 if 'driver_id' in tele_all.columns else tele_all.copy()
        
        required = ['speed_kmh', 'LapDist', 'brakeStatus', 'accStatus']
        missing  = [c for c in required if c not in df_drv.columns]
        if missing:
            continue
        
        enriched = enrich_corners(corners_v3, df_drv, steer_info)
        w     = enriched['confidence'].fillna(0).values
        valid = enriched[enriched['confidence'] > 0]
        
        if len(valid) == 0:
            continue
        
        record = {
            'driver_id': driver_id, 'track': track_name,
            'n_corners_total': len(enriched), 'n_corners_valid': len(valid),
            # ── Mevcut metrikler (v2) ──
            'mean_apex_speed':     wavg(enriched['seg_apex_speed'], w),
            'mean_entry_speed':    wavg(enriched['seg_entry_speed'], w),
            'mean_exit_speed':     wavg(enriched['seg_exit_speed'], w),
            'speed_loss_eff':      wavg(enriched['seg_speed_loss_eff'], w),
            'mean_braking_dist':   wavg(enriched['seg_braking_dist'], w),
            'mean_brake_pressure': wavg(enriched['seg_avg_brake_pressure'], w),
            'mean_coasting_dist':  wavg(enriched['seg_coasting_dist'], w),
            'trail_braking_ratio': float(valid['seg_trail_braking'].mean()),
            'mean_trail_pressure': wavg(enriched['seg_trail_pressure'], w),
            'apex_speed_std':      float(valid['seg_apex_speed'].std()) if len(valid) > 1 else np.nan,
            'braking_dist_std':    float(valid['seg_braking_dist'].std()) if len(valid) > 1 else np.nan,
            'exit_speed_std':      float(valid['seg_exit_speed'].std()) if len(valid) > 1 else np.nan,
            'speed_loss_eff_std':  float(valid['seg_speed_loss_eff'].std()) if len(valid) > 1 else np.nan,
            'pct_heavy_braking':   float((valid['character_class'] == 'heavy_braking').mean()) if 'character_class' in valid.columns else np.nan,
            'pct_trail_braking':   float((valid['character_class'] == 'trail_braking').mean()) if 'character_class' in valid.columns else np.nan,
            'pct_lift_coast':      float((valid['character_class'] == 'lift_coast').mean()) if 'character_class' in valid.columns else np.nan,
            'pct_flat_out':        float((valid['character_class'] == 'flat_out').mean()) if 'character_class' in valid.columns else np.nan,
            # ── YENİ: Faz aggregate metrikleri (v3) ──
            'mean_slb_dist':          wavg(enriched['phase_slb_dist'], w),
            'mean_slb_decel':         wavg(enriched['phase_slb_decel_rate'], w),
            'mean_ce_dist':           wavg(enriched['phase_ce_dist'], w),
            'mean_ce_brake_turnin':   wavg(enriched['phase_ce_brake_at_turnin'], w),
            'mean_mc_speed_ratio':    wavg(enriched['phase_mc_speed_ratio'], w),
            'mean_mc_lateral':        wavg(enriched['phase_mc_lateral_signal'], w),
            'mean_cex_throttle_lag':  wavg(enriched['phase_cex_throttle_lag'], w),
            'mean_cex_accel_rate':    wavg(enriched['phase_cex_accel_rate'], w),
        }
        
        # Viraj bazında detay (mevcut)
        for _, crow in enriched.iterrows():
            t = crow.get('corner_id', '?')
            c = crow.get('confidence', 0)
            record[f'T{t}_apex_speed']   = crow['seg_apex_speed']    if c > 0 else np.nan
            record[f'T{t}_braking_dist'] = crow['seg_braking_dist']  if c > 0 else np.nan
            record[f'T{t}_exit_speed']   = crow['seg_exit_speed']    if c > 0 else np.nan
            record[f'T{t}_trail']        = crow['seg_trail_braking'] if c > 0 else np.nan
        
        all_records.append(record)
        meta_records.append({
            'driver_id': driver_id, 'track': track_name,
            'n_samples': len(df_drv), 'duration_s': len(df_drv) / 50.0,
            'mean_speed': float(df_drv['speed_kmh'].mean()),
        })
    
    mat  = pd.DataFrame(all_records)
    meta = pd.DataFrame(meta_records)
    return mat, meta


# Çalıştır ve kaydet
for track_name in TRACKS:
    if track_name not in loaded:
        continue
    
    print(f"\n{'='*50}")
    print(f"  {track_name.upper()}")
    print(f"{'='*50}")
    
    mat, meta = build_driver_matrix_v3(track_name)
    if mat is not None:
        mat_path  = FEATURES_DIR / f"driver_corner_matrix_{track_name}.parquet"
        meta_path = FEATURES_DIR / f"driver_meta_{track_name}.parquet"
        mat.to_parquet(mat_path, index=False)
        meta.to_parquet(meta_path, index=False)
        
        n_phase = len([c for c in mat.columns if c.startswith('mean_') and ('slb' in c or 'ce_' in c or 'mc_' in c or 'cex' in c)])
        print(f"  {len(mat)} sürücü × {len(mat.columns)} özellik (yeni faz: {n_phase})")
        print(f"  Kaydedildi: {mat_path.name}")

## ADIM 6 — Özet

In [ ]:
# === v4 TESHIS: Exit method dagilimi + korelasyon kontrolu ===

print("=" * 60)
print("  v4 EXIT METHOD TESHIS RAPORU")
print("=" * 60)

for track_name in TRACKS:
    if track_name not in all_enriched:
        continue
    df = all_enriched[track_name]
    valid = df[df['confidence'] > 0]

    print(f"\n  {track_name.upper()}:")

    # Exit method dagilimi
    if 'seg_exit_method' in valid.columns:
        methods = valid['seg_exit_method'].value_counts()
        total = len(valid)
        for m, c in methods.items():
            print(f"    {m:25s}: {c:4d} ({100*c/total:5.1f}%)")

    # apex vs exit std korelasyon
    mat_path = FEATURES_DIR / f'driver_corner_matrix_{track_name}.parquet'
    if mat_path.exists():
        mat = pd.read_parquet(mat_path)
        if 'apex_speed_std' in mat.columns and 'exit_speed_std' in mat.columns:
            a = mat['apex_speed_std'].dropna()
            e = mat['exit_speed_std'].dropna()
            common = a.index.intersection(e.index)
            if len(common) > 2:
                r = np.corrcoef(a[common].values, e[common].values)[0, 1]
                print(f"    apex_std <-> exit_std r = {r:.6f}")
                if abs(r) > 0.999:
                    print(f"    !! HALA r~1.0 — kontrol et")
                elif abs(r) > 0.95:
                    print(f"    ~ Yuksek ama bagimsiz (fizik)")
                else:
                    print(f"    OK — bagimsiz olcum dogrulandi")

print("\n" + "=" * 60)
print("  Hedef: r < 0.95 = bagimsiz olcum")
print("=" * 60)


In [ ]:
print("\n" + "=" * 60)
print("  08_corner_phase_segmentation v3 — ÖZET")
print("=" * 60)

for track_name in TRACKS:
    mat_path = FEATURES_DIR / f"driver_corner_matrix_{track_name}.parquet"
    if mat_path.exists():
        mat = pd.read_parquet(mat_path)
        phase_cols = [c for c in mat.columns if 'slb' in c or 'ce_' in c or 'mc_' in c or 'cex' in c]
        print(f"\n  {track_name.upper()}: {len(mat)} sürücü, {len(mat.columns)} özellik")
        print(f"    Yeni faz kolonları: {phase_cols}")
        
        # Faz metrik örnekleri
        for pc in phase_cols:
            vals = mat[pc].dropna()
            if len(vals) > 0:
                print(f"    {pc}: mean={vals.mean():.3f}, std={vals.std():.3f}, range=[{vals.min():.3f}, {vals.max():.3f}]")

print(f"\n✅ v3 tamamlandı.")
print("Sonraki: 09_driver_style_profiling → faz özelliklerini boyutlara entegre et")